# KASA-42 — Kaggle smoke test

**Purpose: break things here so they cannot break on the H200.**

This runs the *same* `src/kasa42/` code the H200 will run. Nothing here is a
special Kaggle version — only the numbers passed in differ (fewer languages,
smaller batches, fewer steps). Every bug this finds is a bug that would
otherwise have eaten part of a 48-hour window.

Two differences from Thursday, both handled automatically:

| | Kaggle T4 / P100 | H200 |
|---|---|---|
| Mixed precision | **fp16** + GradScaler | **bf16** |
| VRAM | 16 GB | 141 GB |

`train.py` calls `torch.cuda.is_bf16_supported()` and picks. T4 and P100 are
pre-Ampere and have no bf16, so a hardcoded bf16 would fail here — that is
exactly the class of bug this notebook exists to surface.

**Before running:** Settings → Accelerator → **GPU T4 x2**, and Internet → **On**.

## 0 · Get the code

Use the git clone if the repo is pushed; otherwise upload `src/` as a Kaggle Dataset and use the fallback.

In [ ]:
import os, sys, subprocess, pathlib

REPO_URL = 'https://github.com/NasamuAlhassan/kasa42.git'
WORK = pathlib.Path('/kaggle/working/kasa42')

if WORK.exists():
    subprocess.run(['git', '-C', str(WORK), 'pull', '--ff-only'], check=False)
else:
    r = subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(WORK)],
                       capture_output=True, text=True)
    print(r.stdout or '', r.stderr or '')

if not WORK.exists():
    # Fallback if the repo is private: add src/ as a Kaggle Dataset named 'kasa42-src'
    import shutil
    src = pathlib.Path('/kaggle/input/kasa42-src')
    assert src.exists(), 'Clone failed. Make the repo public, or add it as a Kaggle Dataset.'
    shutil.copytree(src, WORK)

os.chdir(WORK)
if str(WORK / 'src') not in sys.path:
    sys.path.insert(0, str(WORK / 'src'))

# Drop any already-imported kasa42 modules. Without this, a `git pull` leaves
# the kernel running the OLD bytecode while tracebacks show the NEW source —
# which is baffling to debug and wastes real time.
for name in [m for m in sys.modules if m.startswith('kasa42')]:
    del sys.modules[name]

print('cwd', os.getcwd())
print('rev', subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                            capture_output=True, text=True).stdout.strip())
print(sorted(p.name for p in (WORK / 'src' / 'kasa42').iterdir()))

In [ ]:
# Kaggle preloads torch/transformers. No -U: upgrading preloaded packages
# forces a kernel restart, which silently discards the os.chdir above.
!pip install -q jiwer onnx onnxruntime 2>&1 | tail -2

import os, sys, torch, transformers
os.environ['HF_HUB_DISABLE_XET'] = '1'
if 'src' not in sys.path:
    sys.path.insert(0, 'src')

# Optional: Add-ons -> Secrets -> HF_TOKEN lifts Hub rate limits.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded')
except Exception:
    print('no HF_TOKEN (optional)')

from kasa42.asr.train import has_native_bf16
import psutil
print(f'\ntorch {torch.__version__} | transformers {transformers.__version__}')
print(f'RAM {psutil.virtual_memory().total/1e9:.0f} GB')
if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability()
    print(f'{torch.cuda.get_device_name(0)}  sm_{cc[0]}{cc[1]}  '
          f'{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB VRAM')
    print(f'native bf16 {has_native_bf16()} -> '
          f'{"bf16" if has_native_bf16() else "fp16"}')
else:
    print('no GPU — sections 1-2 still work; 5-7 need one')

## 1 · Build the manifest here, not at home

Reading metadata from all 533 shards is purely network-bound. On a domestic
connection it takes hours and dies on every blip; on Kaggle it is minutes.

It writes one parquet per config to `results/manifest_parts/`, so a failure
costs one config and a rerun resumes. **Download `results/manifest.parquet` from
the notebook output afterwards** — splits, vocab and mixture then run locally in
seconds, and those four files are the frozen inputs Thursday needs.

`HF_HUB_DISABLE_XET=1` because HF's Xet CAS backend has been throwing
reconstruction errors on large files. Plain HTTP is slower but does not fail.

In [ ]:
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

# Re-runnable: completed configs are cached in results/manifest_parts/ and
# skipped. If this cell dies, just run it again.
!HF_HUB_DISABLE_XET=1 python -m kasa42.data.build_manifest --workers 4

In [ ]:
!python -m kasa42.data.splits
!python -m kasa42.data.vocab
# 400 h x 2 epochs, not 700 x 3 — the T4 benchmark projected ~26 h on the H200.
!python -m kasa42.data.mixture --alpha 0.5 --cap-hours 40 --budget-hours 400

import pathlib
need = ['results/manifest.parquet', 'results/splits.json',
        'results/vocab.json', 'results/mixture.json']
missing = [p for p in need if not pathlib.Path(p).exists()]
if missing:
    print('\nMISSING:', missing)
    print('Re-run the manifest cell — finished configs are skipped.')
else:
    print()
    for p in need:
        print(f'ok  {p:32s} {pathlib.Path(p).stat().st_size/1e6:8.2f} MB')
    print('\nData pipeline done. Save Version (Quick Save) to keep these,')
    print('then download them and commit to the repo.')

## 1 · Logic checks

Runs the full pipeline test: real audio decode, DONDO load, blank index 33,
CTC head transfer, forward/backward, ONNX. Fix anything red before moving on.

In [ ]:
!python tests/test_text.py

In [ ]:
!python tests/test_pipeline.py

import os, pathlib, sys
if not pathlib.Path('src/kasa42').exists():
    os.chdir('/kaggle/working/kasa42')

SMOKE_LANGS = ['Kusaal_kus', 'Asante_Twi_twi', 'Ewe_ewe', 'Dagaare_dga', 'Mampruli_maw']

# Subprocess so HF_HUB_DISABLE_XET actually applies — setting it in-process is
# too late once huggingface_hub has been imported.
pats = ','.join(f'{c}/train-00000-*' for c in SMOKE_LANGS)
!HF_HUB_DISABLE_XET=1 python -c "\
from huggingface_hub import snapshot_download; \
snapshot_download('ghananlpcommunity/ghana-speech', repo_type='dataset', \
                  local_dir='data/parquet', allow_patterns='$pats'.split(','), \
                  max_workers=4); print('download ok')"

!du -sh data/parquet

In [ ]:
import os, pathlib, sys
# Self-contained: a kernel restart wipes variables from earlier cells, and this
# cell should not fail with a NameError for that reason.
if not pathlib.Path('src/kasa42').exists():
    os.chdir('/kaggle/working/kasa42')
if 'src' not in sys.path:
    sys.path.insert(0, 'src')

SMOKE_LANGS = ['Kusaal_kus', 'Asante_Twi_twi', 'Ewe_ewe', 'Dagaare_dga', 'Mampruli_maw']

from kasa42.asr.train import train, TrainConfig

# T4 settings. The H200 needs neither: batch_duration goes to 320+, and
# gradient checkpointing is unnecessary when optimiser state is 7% of VRAM
# rather than 60%.
ckpt = train(TrainConfig(smoke=True, languages=SMOKE_LANGS,
                         batch_duration=60.0,
                         gradient_checkpointing=True,
                         out_dir='checkpoints/smoke'))
print('checkpoint:', ckpt)

## 3 · A real (tiny) training run

Same `train()` the H200 calls. Five languages spanning the size range —
Asante Twi (200 h) down to Mampruli (~5 h) — so the mixture and bucketing meet
the same imbalance they will on Thursday.

Needs the parquet for those configs; ~2 GB, well inside Kaggle's disk.

In [ ]:
SMOKE_LANGS = ['Kusaal_kus', 'Asante_Twi_twi', 'Ewe_ewe', 'Dagaare_dga', 'Mampruli_maw']

from huggingface_hub import snapshot_download
snapshot_download('ghananlpcommunity/ghana-speech', repo_type='dataset',
                  local_dir='data/parquet',
                  allow_patterns=[f'{c}/train-00000-*' for c in SMOKE_LANGS],
                  max_workers=8)
!du -sh data/parquet

In [ ]:
from kasa42.asr.train import train, TrainConfig

# batch_duration 120 for 16 GB; the H200 default is 320 and can go far higher.
ckpt = train(TrainConfig(smoke=True, languages=SMOKE_LANGS,
                         batch_duration=120.0, out_dir='checkpoints/smoke'))
print('checkpoint:', ckpt)

## 4 · Baselines — do they even load and decode?

Correctness of the plumbing, not the numbers. 30 steps of training predicts nothing.

In [ ]:
import torch
from transformers import AutoModelForCTC, AutoProcessor
from kasa42.asr.baselines import DONDO, DONDO_LANGS, iso_of
from kasa42.asr.dataset import decode_audio

proc = AutoProcessor.from_pretrained(DONDO)
m = AutoModelForCTC.from_pretrained(DONDO).cuda().eval()

import pyarrow.parquet as pq, glob
f = sorted(glob.glob('data/parquet/Kusaal_kus/*.parquet'))[0]
rows = next(pq.ParquetFile(f).iter_batches(batch_size=4)).to_pylist()
wavs = [decode_audio(r['audio']) for r in rows]

inp = proc(wavs, sampling_rate=16000, return_tensors='pt', padding=True).to('cuda')
with torch.no_grad():
    logits = m(**inp).logits
hyp = proc.batch_decode(logits.argmax(-1).cpu().numpy())
hyp = hyp.text if hasattr(hyp, 'text') else hyp
for r, h in zip(rows, hyp):
    print('REF', r['text'][:70]); print('DONDO', h[:70]); print()

## 5 · Export path — the demo must outlive the GPU

In [ ]:
!python -m kasa42.asr.export --checkpoint checkpoints/smoke/final.pt \
    --model-config checkpoints/smoke/config.json --out-dir export_smoke

# The check that matters: does it run with the GPU hidden?
!CUDA_VISIBLE_DEVICES='' KASA42_EXPORT=export_smoke KASA42_MODE=onnx python -c "\
import sys; sys.path.insert(0,'src'); import numpy as np; \
from kasa42.app.app import Engine; e=Engine('onnx'); \
t,l,c,dt=e.transcribe(16000, np.zeros(32000,dtype=np.float32)); \
print('mode', e.mode, '| lang', l, '|', f'{dt*1000:.0f}ms')"

## 6 · What did we learn?

Write down anything that failed and fix it in `src/`, then re-run from the top.
Do not carry a known failure into Thursday.

Checklist before closing this notebook:

- [ ] `test_text.py` and `test_pipeline.py` fully green
- [ ] `bench_gpu.py` gives an H200 estimate under ~12 h
- [ ] `train(smoke=True)` completes and the loss moves
- [ ] DONDO baseline produces plausible Kusaal text
- [ ] ONNX export runs **with `CUDA_VISIBLE_DEVICES=''`**
- [ ] Largest `batch_duration` that fits on 16 GB is recorded — the H200 has
      141 GB, so Thursday's value can be far larger

In [ ]:
import json, pathlib
p = pathlib.Path('results/bench_gpu.json')
if p.exists():
    print(json.dumps(json.loads(p.read_text()), indent=2))